## Paso 0 — Sello de trazabilidad del insumo crudo

Registramos el hash MD5 del archivo crudo y lo comparamos contra el valor
esperado (calculado la primera vez que se corrió este notebook). Si el
archivo cambia, esta celda debe fallar — es la señal de que alguien
reemplazó el insumo.

In [30]:
import hashlib
from datetime import datetime
from pathlib import Path

RAW_DIR = Path("../datos")
archivo_crudo = RAW_DIR / "Viajes-Btura-2025-2026.xlsx"

hash_md5 = hashlib.md5(archivo_crudo.read_bytes()).hexdigest()

HASH_ESPERADO = "9102c305227a8c37ae6939f643c4861a"  # calculado el 2026-09-08
assert hash_md5 == HASH_ESPERADO, (
    f"¡El archivo crudo cambió! Esperado: {HASH_ESPERADO}, actual: {hash_md5}"
)

print(f"Archivo:     {archivo_crudo.name}")
print(f"Hash MD5:    {hash_md5}  (coincide con lo esperado ✓)")
print(f"Ejecutado:   {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Archivo:     Viajes-Btura-2025-2026.xlsx
Hash MD5:    9102c305227a8c37ae6939f643c4861a  (coincide con lo esperado ✓)
Ejecutado:   2026-09-08 08:53:13


## Paso 1 — Inspección defensiva de Viajes-Btura (el dataset del EDA)

Revisamos cuántas filas y columnas tiene `Hoja1` **de verdad** antes de
cargarla, usando openpyxl directamente (sin pasar por pandas). Este es el
archivo sobre el que corre todo el análisis exploratorio.

In [31]:
import openpyxl

wb = openpyxl.load_workbook(RAW_DIR / "Viajes-Btura-2025-2026.xlsx", read_only=True, data_only=True)
for hoja in wb.sheetnames:
    ws = wb[hoja]
    print(f"hoja '{hoja}': max_row={ws.max_row}  max_col={ws.max_column}")

hoja 'Hoja1': max_row=41238  max_col=17


## Paso 2 — Inspección del archivo de inconsistencias (apoyo, no es sujeto de EDA)

Este archivo no se analiza por sí mismo. Sirve como diccionario de columnas
(hoja `Distancia_0`) y para validación cruzada más adelante. Lo inspeccionamos
con el mismo cuidado que en el Paso 1, pero aquí no basta con `max_row`: ya
sabemos que trae relleno de formato en filas vacías.


In [32]:
def ultima_fila_con_contenido(ws):
    ultima = 0
    for fila in ws.iter_rows():
        if any(celda.value is not None for celda in fila):
            ultima = fila[0].row
    return ultima

wb = openpyxl.load_workbook(RAW_DIR / "inconsistencias_viajes.xlsx", read_only=False, data_only=True)
for hoja in wb.sheetnames:
    ws = wb[hoja]
    real = ultima_fila_con_contenido(ws)
    print(f"hoja '{hoja}': max_row={ws.max_row}  ultima_fila_con_contenido={real}")


hoja 'Distancia_0': max_row=1001  ultima_fila_con_contenido=31
hoja 'Costo_0': max_row=1000  ultima_fila_con_contenido=875
hoja 'ImporteFO_0': max_row=1000  ultima_fila_con_contenido=184
hoja 'Peso_Neto_0': max_row=1000  ultima_fila_con_contenido=83
hoja 'ImporteFO_Negativo': max_row=1000  ultima_fila_con_contenido=2


## Paso 3 — Cargamos Viajes-Btura, todo como texto por ahora

Traemos las 41.237 filas con pandas, pero le pedimos que no invente ningún
tipo todavía (`dtype=str`) — así vemos el dato tal cual llega, antes de
decidir nosotros mismos, con cuidado, cómo convertir cada columna.

In [33]:
import pandas as pd

df_crudo = pd.read_excel(RAW_DIR / "Viajes-Btura-2025-2026.xlsx", sheet_name="Hoja1", dtype=str)

print(df_crudo.shape)
df_crudo.head()


(41237, 17)


,ID Clase Documento FWO,FO,Fecha Creación FO,BP Cliente,Origen,Destino,Distancia Total,Peso Neto,Posición Principal FWOP,Grupo de Equipo FWOP,Vehículo,ID Medio Transporte FO,Clase Equipo FOP,Propiedad,Tipo,Tarifa venta al cliente,Flete producción FO
0,OPIM,301403469,2026-03-31 00:00:00,1,BUENAVENTURA,GIRON,712,7970,TUR,CN,TTX595,TC-2S2,CNT40.HC,Tercero,TC.2S2,5800000,5800000
1,OPTN,301269707,2025-07-31 00:00:00,100229,BUENAVENTURA,PUERTO GAITAN,826,34000,TUR,CAM,SSQ288,TC-3S3,3S3.34T,Tercero,TC.3S3,9860000,8874000
2,OPTN,301272820,2025-08-06 00:00:00,100229,BUENAVENTURA,PUERTO GAITAN,826,34000,TUR,CAM,SXV119,TC-3S3,3S3.34T,Tercero,TC.3S3,10200000,9180000
3,OPTN,301276538,2025-08-14 00:00:00,100229,BUENAVENTURA,TURBO,788,34000,TUR,CAM,SWK320,TC-3S3,3S3.34T,Tercero,TC.3S3,8537500,8000000
4,OPTN,301277447,2025-08-15 00:00:00,100229,BUENAVENTURA,TURBO,788,9800,TUR,CAM,WEB025,TC_SENCI,C2.8T,Tercero,C2.8T,3400000,3400000


## Paso 4 — Convertimos cada columna a su tipo real, contando lo que falla

Fechas a fecha, números a número, texto sin espacios de más. Por cada
columna que convertimos, contamos cuántos valores no se pudieron convertir
— así cualquier problema queda a la vista como un número, no escondido en
silencio.

También revisamos, antes de convertir `BP Cliente` y `FO` a número, que
ningún código tenga un cero a la izquierda (se perdería sin aviso). Y las 4
columnas de plata/medidas las forzamos a decimal (`float64`) a propósito,
para que no quede en manos de pandas decidir solo.

In [34]:
import numpy as np

def contar_fallos(original, convertido, nombre):
    no_nulo_antes = original.notna()
    nulo_despues = convertido.isna()
    fallos = (no_nulo_antes & nulo_despues).sum()
    print(f"  {nombre}: {fallos} valor(es) no se pudieron convertir (de {no_nulo_antes.sum()} no nulos)")

df_tipado = df_crudo.copy()

# --- texto: solo recortamos espacios al inicio/fin ---
columnas_texto = [
    "ID Clase Documento FWO", "Origen", "Destino", "Posición Principal FWOP",
    "Grupo de Equipo FWOP", "Vehículo", "ID Medio Transporte FO",
    "Clase Equipo FOP", "Propiedad", "Tipo",
]
for col in columnas_texto:
    df_tipado[col] = df_crudo[col].str.strip()

# --- fecha ---
df_tipado["Fecha Creación FO"] = pd.to_datetime(df_crudo["Fecha Creación FO"], errors="coerce")
contar_fallos(df_crudo["Fecha Creación FO"], df_tipado["Fecha Creación FO"], "Fecha Creación FO")

# --- numéricas: forzamos float64 a propósito ---
columnas_numericas = ["Distancia Total", "Peso Neto", "Tarifa venta al cliente", "Flete producción FO"]
for col in columnas_numericas:
    df_tipado[col] = pd.to_numeric(df_crudo[col], errors="coerce").astype(float)
    contar_fallos(df_crudo[col], df_tipado[col], col)

# --- BP Cliente: chequeo ANTES de convertir, para no perder un cero a la izquierda sin darnos cuenta ---
tiene_cero_inicial = df_crudo["BP Cliente"].str.match(r"^0\d").fillna(False)
print(f"BP Cliente con cero a la izquierda: {tiene_cero_inicial.sum()}")
df_tipado["BP Cliente"] = pd.to_numeric(df_crudo["BP Cliente"], errors="coerce").astype("Int64")
contar_fallos(df_crudo["BP Cliente"], df_tipado["BP Cliente"], "BP Cliente")

# --- FO: los 130 repetidos NO se tocan acá, eso se investiga en la fase de unicidad (§4 del plan) ---
fo_con_cero_inicial = df_crudo["FO"].str.match(r"^0\d").fillna(False)
print(f"FO con cero a la izquierda: {fo_con_cero_inicial.sum()}")
df_tipado["FO"] = pd.to_numeric(df_crudo["FO"], errors="coerce").astype("Int64")
contar_fallos(df_crudo["FO"], df_tipado["FO"], "FO")

print("\nTipos finales:")
print(df_tipado.dtypes)


  Fecha Creación FO: 0 valor(es) no se pudieron convertir (de 41237 no nulos)
  Distancia Total: 0 valor(es) no se pudieron convertir (de 41237 no nulos)
  Peso Neto: 0 valor(es) no se pudieron convertir (de 41237 no nulos)
  Tarifa venta al cliente: 0 valor(es) no se pudieron convertir (de 41237 no nulos)
  Flete producción FO: 0 valor(es) no se pudieron convertir (de 41237 no nulos)
BP Cliente con cero a la izquierda: 0
  BP Cliente: 0 valor(es) no se pudieron convertir (de 41237 no nulos)
FO con cero a la izquierda: 0
  FO: 0 valor(es) no se pudieron convertir (de 41237 no nulos)

Tipos finales:
ID Clase Documento FWO                str
FO                                  Int64
Fecha Creación FO          datetime64[us]
BP Cliente                          Int64
Origen                                str
Destino                               str
Distancia Total                   float64
Peso Neto                         float64
Posición Principal FWOP               str
Grupo de Equipo 

## Paso 5 — Normalizamos nombres de columna a camelCase

Convertimos los encabezados a camelCase (primera palabra en minúscula, cada
palabra siguiente con su inicial en mayúscula, sin espacios ni tildes) para
que el código de los próximos notebooks sea menos frágil. Guardamos el
mapeo original → nuevo para no perder la trazabilidad hacia el nombre
"humano" que se usa en los reportes al negocio.


In [35]:
import re
import unicodedata

def a_camel_case(nombre):
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode("ascii")
    palabras = [p for p in re.split(r"[^a-zA-Z0-9]+", sin_tildes) if p]
    primera, *resto = palabras
    return primera.lower() + "".join(p.capitalize() for p in resto)

mapeo_columnas = {col: a_camel_case(col) for col in df_tipado.columns}

# defensivo: si dos columnas distintas terminaran generando el mismo nombre nuevo, queremos enterarnos aquí
nombres_nuevos = list(mapeo_columnas.values())
assert len(nombres_nuevos) == len(set(nombres_nuevos)), "Colisión: dos columnas producen el mismo camelCase"

for original, nuevo in mapeo_columnas.items():
    print(f"{original!r:35} -> {nuevo}")

df_tipado = df_tipado.rename(columns=mapeo_columnas)


'ID Clase Documento FWO'            -> idClaseDocumentoFwo
'FO'                                -> fo
'Fecha Creación FO'                 -> fechaCreacionFo
'BP Cliente'                        -> bpCliente
'Origen'                            -> origen
'Destino'                           -> destino
'Distancia Total'                   -> distanciaTotal
'Peso Neto'                         -> pesoNeto
'Posición Principal FWOP'           -> posicionPrincipalFwop
'Grupo de Equipo FWOP'              -> grupoDeEquipoFwop
'Vehículo'                          -> vehiculo
'ID Medio Transporte FO'            -> idMedioTransporteFo
'Clase Equipo FOP'                  -> claseEquipoFop
'Propiedad'                         -> propiedad
'Tipo'                              -> tipo
'Tarifa venta al cliente'           -> tarifaVentaAlCliente
'Flete producción FO'               -> fleteProduccionFo


## Paso 6 — Guardamos el dataset ya limpio, para no repetir este trabajo

Guardamos `df_tipado` (ya tipado, verificado y con columnas en camelCase) en
`datos/interim/`, en formato Parquet (Conserva los tipos exactos ya definidos). Los próximos notebooks van a abrir
directamente este archivo, sin tener que volver a leer el Excel ni repetir
la conversión.


In [36]:
df_tipado.to_parquet("../datos/interim/viajes_btura_tipado.parquet", index=False)
print("Guardado en datos/interim/viajes_btura_tipado.parquet")
print(df_tipado.shape)


Guardado en datos/interim/viajes_btura_tipado.parquet
(41237, 17)
